In [1]:
import sys
sys.path.append('/Users/brianlerner/Library/CloudStorage/OneDrive-Personal/Code/cerebra/src')
from server import connect_to_collection

from datetime import datetime

from server.query import get_n_occurrences

n = get_n_occurrences("Basketball Game")
n



18

In [5]:
events[0]

{'_id': ObjectId('6580954613c558ff3d413722'),
 'calendar_events': [{'name': 'Basketball Game',
   'id': 'basketball_game',
   'group': 'basketball_team',
   'location': 'Cameron Indoor Stadium',
   'description': 'Game for the Duke basketball team.',
   'start_datetime': datetime.datetime(2023, 9, 1, 19, 0),
   'end_datetime': datetime.datetime(2023, 9, 1, 22, 0)},
  {'name': 'Basketball Game',
   'id': 'basketball_game',
   'group': 'basketball_team',
   'location': 'Cameron Indoor Stadium',
   'description': 'Game for the Duke basketball team.',
   'start_datetime': datetime.datetime(2023, 9, 8, 19, 0),
   'end_datetime': datetime.datetime(2023, 9, 8, 22, 0)},
  {'name': 'Basketball Game',
   'id': 'basketball_game',
   'group': 'basketball_team',
   'location': 'Cameron Indoor Stadium',
   'description': 'Game for the Duke basketball team.',
   'start_datetime': datetime.datetime(2023, 9, 15, 19, 0),
   'end_datetime': datetime.datetime(2023, 9, 15, 22, 0)},
  {'name': 'Basketball G

In [ ]:

collection = connect_to_collection()

def get_schema(collection):

    def add_name_to_schema(event, schema, cat):
        if 'name' not in event:
            pass
        else:
            if cat not in schema:
                schema[cat] = []
            if event['name'] not in schema[cat]:
                schema[cat].append(event['name'])
                
    schema = {}
    document = next(collection.find({})) # should only be one entry
    for cat,v in document.items():
        if type(v) == list:
            for event in v:
                add_name_to_schema(event, schema, cat)

    mapping = {}
    for k,v in schema.items():
        for v in v:
            mapping[v] = k
    mapping

    return mapping

def get_events(
    start_date=None, 
    end_date=None, 
    event_name=None, 
    # cat=None,
    # group=None, 
    profile='llm_v0', 
    print_events=False
    ):
    

    schema = get_schema(collection)
    if event_name in schema:
        cat = schema[event_name]
    else:
        print(f'event_name: {event_name} not found in schema')
        return []
    # conditions = [
    #                 {"$gte": ["$$event.start_datetime", start_date]},
    #                 {"$lte": ["$$event.start_datetime", end_date]},
    #             ]
    conditions = []
    if start_date:
        conditions.append({"$gte": ["$$event.start_datetime", start_date]})
    if end_date:
        conditions.append({"$lte": ["$$event.start_datetime", end_date]})
    if event_name:
        conditions.append({"$eq": ["$$event.name", event_name]})
    # if group:
        # conditions.append({"$eq": ["$$event.group", group]})

    pipeline = [
        {"$match": {"profile_id": profile}},  # Filter to the specific profile
        {"$project": {  # Project only the calendarEvents field
            "calendar_events": {
                "$filter": {  # Filter the calendarEvents array
                    "input": f"${cat}",
                    "as": "event",
                    "cond": {  # Condition to check the date range
                        "$and": conditions
                    }
                }
            }
        }}
    ]

    # Execute the aggregation pipeline
    cursor = collection.aggregate(pipeline)
    events_list = [event for event in cursor]

    if print_events:
        # Process and print the results
        for events in events_list:
            # print(events)
            print()
            for event in events.get("calendar_events", []):
                print(event)


    return events_list

In [21]:
get_events(event_name='Basketball Game')

[{'_id': ObjectId('6580954613c558ff3d413722'),
  'calendar_events': [{'name': 'Basketball Game',
    'id': 'basketball_game',
    'group': 'basketball_team',
    'location': 'Cameron Indoor Stadium',
    'description': 'Game for the Duke basketball team.',
    'start_datetime': datetime.datetime(2023, 9, 1, 19, 0),
    'end_datetime': datetime.datetime(2023, 9, 1, 22, 0)},
   {'name': 'Basketball Game',
    'id': 'basketball_game',
    'group': 'basketball_team',
    'location': 'Cameron Indoor Stadium',
    'description': 'Game for the Duke basketball team.',
    'start_datetime': datetime.datetime(2023, 9, 8, 19, 0),
    'end_datetime': datetime.datetime(2023, 9, 8, 22, 0)},
   {'name': 'Basketball Game',
    'id': 'basketball_game',
    'group': 'basketball_team',
    'location': 'Cameron Indoor Stadium',
    'description': 'Game for the Duke basketball team.',
    'start_datetime': datetime.datetime(2023, 9, 15, 19, 0),
    'end_datetime': datetime.datetime(2023, 9, 15, 22, 0)},
  

In [19]:




# Use the function for your database
schema = get_schema_mapping(collection)




{'elbow pain': 'extracted',
 'Basketball Game': 'calendar_events',
 'Computer Programming': 'calendar_events',
 'Basketball Practice': 'calendar_events',
 'Discrete Mathematics': 'calendar_events',
 'Discrete Mathematics HW due': 'calendar_events',
 'Computer Programming HW due': 'calendar_events'}